### env

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_cwd = Path.cwd().resolve()
QUANT_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents) if (candidate / 'project_paths.py').is_file()),
    Path(__file__).resolve().parents[2] if '__file__' in globals() else _cwd,
)
if str(QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(QUANT_ROOT))
from project_paths import GITHUB_ROOT, NOTE_REPO_ROOT

REPO_ROOT = GITHUB_ROOT
NOTE_ROOT = NOTE_REPO_ROOT
PROJECT_DIR = REPO_ROOT / 'projects/workStrategy/TX'
for path in (REPO_ROOT, NOTE_ROOT, PROJECT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from analyzer import TXAnalyzer
from cloud_data import read_tx_futures

START = '2020-01-01'
END = datetime.now().strftime('%Y-%m-%d')
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.00  # Set to 0 for a simple train/test backtest.
ONE_WAY_COST = 0.00002

tx_futures = read_tx_futures(START, END)
analyzer = TXAnalyzer(tx_futures)

split_dates = TXAnalyzer.split_periods(
    analyzer.display_df().index,
    start=START,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
)
TRAIN_END = split_dates['train_end']
training_analyzer = analyzer.for_period(end=TRAIN_END)

display(analyzer.session_alignment_report())
pd.Series(split_dates, name='split date')

total_dates                1596
paired_day_night_dates     1588
day_only_dates                0
night_only_dates              8
missing_both_dates            0
duplicate_dates               0
close_date_alignment_ok    True
Name: session_alignment, dtype: object

train_end    2024-08-02
test_start   2024-08-05
Name: split date, dtype: datetime64[us]

### ma_divergence x hist_vol

In [4]:
hist_vol_40d = training_analyzer.indicator_hist_vol(40, return_series=True)
ma_divergence_factors = {
    f'{window}MA divergence': training_analyzer.indicator_ma_divergence(window, return_series=True)
    for window in range(5, 51, 2)
}
training_analyzer.compare_factor_percentiles(
    factors=ma_divergence_factors,
    condition_factor=hist_vol_40d,
    condition_name='HistVol 40D',
    return_column='daily_ret',
    condition_bins=5,
    factor_bin_percentile=15,
    title='日盤：MA divergence × HistVol 40D',
)

ma25_divergence = training_analyzer.indicator_ma_divergence(10, return_series=True)
training_analyzer.conditional_factor_sort(
    conditions=[
        {'factor': hist_vol_40d,
        'percentile_range': (60, 80),
        'name': 'hist_vol_40'}
    ],
    sort_factor=ma25_divergence,
    return_column='daily_ret',
    # plot_mode='bin',
    show_before_sort=True
)

training_analyzer.show_factor_signal_overlap([
    {'factor': ma25_divergence, 'name': 'MA25 / vol21D', 'volatility_window': 21, 'volatility_regime': 4, 'volatility_bins': 5, 'factor_percentile': 15},
    {'factor': ma25_divergence, 'name': 'MA25 / vol40D', 'volatility_window': 40, 'volatility_regime': 4, 'volatility_bins': 5, 'factor_percentile': 15},
])

### night_ret x hist_vol

In [3]:
hist_vol_40d = training_analyzer.indicator_hist_vol(window=40, return_series=True)
night_ret_factors = {
    f'{window} night_ret': training_analyzer.indicator_night_ret(window=window, return_series=True)
    for window in range(5, 51, 5)
}

training_analyzer.compare_factor_percentiles(
    factors=night_ret_factors,
    condition_factor=hist_vol_40d,
    condition_name='HistVol 40D',
    return_column='daily_ret',
    condition_bins=5,
    factor_bin_percentile=5,
    title='日盤：night_ret × HistVol 40D',
)

night_ret_factor = training_analyzer.indicator_night_ret(window=10, return_series=True)
training_analyzer.conditional_factor_sort(
    conditions=[
        {'factor': hist_vol_40d,
        'percentile_range': (0, 80),
        'name': 'hist_vol_40'}
    ],
    sort_factor=night_ret_factor,
    return_column='daily_ret',
    plot_mode='bin',
    show_before_sort=False
)

training_analyzer.show_factor_signal_timeline(
    factor=night_ret_factor,
    factor_name='night_ret',
    factor_percentile=(0, 20),
    conditions=[
        {'factor': hist_vol_40d,
        'percentile_range': (60, 80),
        'name': 'hist_vol_40'}
    ],
    return_column='daily_ret',
    position=-1,
)

,Total Return,CAGR,Volatility,Sharpe,Max Drawdown,Max DD Duration,Profit Factor,Win Rate,Odds,Avg Win,Avg Loss,Avg Return (Exp),Kelly,Annual Turnover
0,0.146009,0.031224,0.042235,0.748785,-0.030871,469.0,2.052066,0.536585,1.772239,0.012428,-0.007013,0.003419,0.2751,9.024172


### night_ret_divergence

In [45]:
HL_vol_40d = training_analyzer.indicator_HL_vol(window=40, return_series=True)
night_ret_factors = {
    f'{window} night_ret': training_analyzer.indicator_night_ret_divergence(window=window, return_series=True)
    for window in range(2, 30, 1)
}

training_analyzer.compare_factor_percentiles(
    factors=night_ret_factors,
    return_column='daily_ret',
    factor_bin_percentile=5,
    condition_factor=HL_vol_40d,
    condition_bins=5,
    title='日盤：night_ret_divergence'
)

night_ret_factor = training_analyzer.indicator_night_ret_divergence(window=3, return_series=True)
training_analyzer.show_factor_signal_timeline(
    factor=night_ret_factor,
    factor_name='night_ret_divergence',
    factor_percentile=(0, 25),
    conditions=[
        {
            'factor': HL_vol_40d,
            'percentile_range': (0, 20),
            'name': 'HL_vol_40'
        }
    ],
    return_column='daily_ret',
    position=1,
    one_way_cost=ONE_WAY_COST,
)

,Total Return,CAGR,Volatility,Sharpe,Max Drawdown,Max DD Duration,Profit Factor,Win Rate,Odds,Avg Win,Avg Loss,Avg Return (Exp),Kelly,Annual Turnover
0,0.039355,0.00877,0.01697,0.523034,-0.020048,770.0,2.277752,0.333333,4.555503,0.009992,-0.002193,0.001868,0.18699,4.512086


### option_pos

### 檢查因子關係

In [3]:
features = pd.concat({
    'day_divergence': analyzer.indicator_ma_divergence(25, return_series=True),
    'night_divergence': analyzer.indicator_night_ret(window=3, return_series=True),
    'hist_vol_40': analyzer.indicator_hist_vol(40, return_series=True),
}, axis=1).reindex(analyzer.display_df().index)
training_features = features.loc[:TRAIN_END]

day_thresholds = TXAnalyzer.fit_factor_thresholds(
    training_features['day_divergence'],
    15,
    condition=training_features['hist_vol_40'],
    condition_percentile_range=(60, 80),
)
night_thresholds = TXAnalyzer.fit_factor_thresholds(
    training_features['night_divergence'],
    20,
    condition=training_features['hist_vol_40'],
    condition_percentile_range=(60, 80),
)
SIGNALS_TO_CHECK = {
    'day_divergence': TXAnalyzer.threshold_signal(
        features['day_divergence'],
        day_thresholds,
        condition=features['hist_vol_40'],
    ),
    'night_divergence': TXAnalyzer.threshold_signal(
        features['night_divergence'],
        night_thresholds,
        condition=features['hist_vol_40'],
    ),
}

EVALUATION_START = START
EVALUATION_END = TRAIN_END
TARGET_RETURN = analyzer.display_df()['daily_ret']

diagnostics = TXAnalyzer.signal_overlap_diagnostics(
    signals=SIGNALS_TO_CHECK,
    forward_returns=TARGET_RETURN,
    evaluation_start=EVALUATION_START,
    evaluation_end=EVALUATION_END,
    lead_lags=(-1, 0, 1),
    hac_lags=5,
)

TXAnalyzer.export_diagnostics(
    diagnostics,
    'rubber_band_diagnostics.xlsx',
)

C:\Users\user\AppData\Local\Temp\ipykernel_1636\246827754.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  features = pd.concat({


WindowsPath('rubber_band_diagnostics.xlsx')

## 回測

In [48]:
features = pd.concat({
    'ma_divergence': analyzer.indicator_ma_divergence(25, return_series=True),
    'night_ret_divergence': analyzer.indicator_night_ret_divergence(window=3, return_series=True),
    'hist_vol_40': analyzer.indicator_hist_vol(40, return_series=True),
}, axis=1).reindex(analyzer.display_df().index)
training_features = features.loc[:TRAIN_END]

ma_divergence_thresholds = TXAnalyzer.fit_factor_thresholds(
    training_features['ma_divergence'],
    (0, 15),
    condition=training_features['hist_vol_40'],
    condition_percentile_range=(60, 80),
)
short_night_ret_divergence_thresholds = TXAnalyzer.fit_factor_thresholds(
    training_features['night_ret_divergence'],
    (90, 100),
)
long_night_ret_divergence_thresholds = TXAnalyzer.fit_factor_thresholds(
    training_features['night_ret_divergence'],
    (0, 25),
)

# Both signals trade the same-date day session. If both fire, exposure remains -1.0.
rules = {
    'short_night_ret_divergence': {
        'factor': 'night_ret_divergence',
        'thresholds': short_night_ret_divergence_thresholds,
        'position': -1.0,
        'session': 'day'
    },
    # 'long_night_ret_divergence': {
    #     'factor': 'night_ret_divergence',
    #     'thresholds': long_night_ret_divergence_thresholds,
    #     'position': 1.0,
    #     'session': 'day'
    # },
}

backtest = analyzer.backtest_threshold_rules(
    features,
    rules,
    split_dates,
    one_way_cost=ONE_WAY_COST,
    title='Rubber band: test-period net equity curve',
    plot_period='both',
)

C:\Users\user\AppData\Local\Temp\ipykernel_15704\1663456910.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  features = pd.concat({


,Total Return,CAGR,Volatility,Sharpe,Max Drawdown,Max DD Duration,Profit Factor,Win Rate,Odds,Avg Win,Avg Loss,Avg Return (Exp),Kelly,Annual Turnover
Train Strategy,0.247438,0.051285,0.055968,0.921278,-0.041015,428.0,1.708626,0.257282,4.932449,0.010370,-0.002102,0.001106,0.106704,42.864816
Train Benchmark,0.837389,0.147960,0.196145,0.802110,-0.315079,507.0,1.153830,0.553345,0.931359,0.008501,-0.009128,0.000627,0.073773,0.000000
Test Strategy,0.073570,0.038463,0.065599,0.607955,-0.083543,377.0,1.280956,0.316176,2.770440,0.007954,-0.002871,0.000552,0.069348,66.288100
Test Benchmark,1.005655,0.451165,0.280193,1.470854,-0.276565,180.0,1.308994,0.556263,1.044198,0.012455,-0.011928,0.001635,0.131309,0.000000
